# RK1828 Paraformer Chinese ASR Demo (with hotwords)


Alibaba [Paraformer](https://github.com/modelscope/FunASR) **non-autoregressive**
Chinese ASR: one forward pass emits the whole transcript (no token-by-token
generation like Whisper). Measured RTF 0.046 (~21x realtime). It also supports
**hotword boosting** — given words are recognized preferentially
(asf_scorer + hotword decoder).

| Step | Content |
|---|---|
| 1 | Download models and sample audio (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Pick audio (hotword mode optional) |
| 4 | Transcribe |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("paraformer-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/paraformer/demo_x86_64.tar.gz"),
    ("paraformer-rknn3-rk1828", "model/asf_scorer.rknn", "demos/paraformer/model/asf_scorer.rknn"),
    ("paraformer-rknn3-rk1828", "model/asf_scorer.weight", "demos/paraformer/model/asf_scorer.weight"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "demos/paraformer/model/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "demos/paraformer/model/asr_example_hotword.wav"),
    ("paraformer-rknn3-rk1828", "model/bias_encoder.bin", "demos/paraformer/model/bias_encoder.bin"),
    ("paraformer-rknn3-rk1828", "model/hotwords.txt", "demos/paraformer/model/hotwords.txt"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder.rknn", "demos/paraformer/model/paraformer_decoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder.weight", "demos/paraformer/model/paraformer_decoder.weight"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder_hotword.rknn", "demos/paraformer/model/paraformer_decoder_hotword.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_decoder_hotword.weight", "demos/paraformer/model/paraformer_decoder_hotword.weight"),
    ("paraformer-rknn3-rk1828", "model/paraformer_encoder.rknn", "demos/paraformer/model/paraformer_encoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/paraformer_encoder.weight", "demos/paraformer/model/paraformer_encoder.weight"),
    ("paraformer-rknn3-rk1828", "model/seaco_decoder.rknn", "demos/paraformer/model/seaco_decoder.rknn"),
    ("paraformer-rknn3-rk1828", "model/seaco_decoder.weight", "demos/paraformer/model/seaco_decoder.weight"),
    ("paraformer-rknn3-rk1828", "model/seg_dict", "demos/paraformer/model/seg_dict"),
    ("paraformer-rknn3-rk1828", "model/tokens.json", "demos/paraformer/model/tokens.json"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/paraformer/demo_x86_64.tar.gz", "demos/paraformer")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Pick audio and mode

- **Normal mode**: `paraformer_encoder` + `paraformer_decoder`
- **Hotword mode**: `paraformer_decoder_hotword` + `asf_scorer` instead,
  boosting the words listed in `model/hotwords.txt`
  (sample hotword audio: datasets/asr/asr_example_hotword.wav)

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="Audio:", layout=widgets.Layout(width="60%"))
if any(o.endswith('/asr_example.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/asr_example.wav')][0]
w_hot = widgets.Checkbox(value=False, description="Hotword mode")
display(w_wav, w_hot)
print("Then keep running downwards ▼")

Listen to the audio first:

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. Transcribe

In [ ]:
import os

DEMO = "demos/paraformer"
wav = os.path.abspath(w_wav.value)
if w_hot.value:
    print(open(os.path.join(DEMO, "model/hotwords.txt")).read())
    run_demo(["./rknn_paraformer_demo",
              "--encoder_path", "model/paraformer_encoder.rknn",
              "--decoder_path", "model/paraformer_decoder_hotword.rknn",
              "--tokens_path", "model/tokens.json",
              "--audio_path", wav,
              "--seaco_path", "model/seaco_decoder.rknn",
              "--hotwords_path", "model/hotwords.txt",
              "--asf_path", "model/asf_scorer.rknn",
              "--asf_nfilter", "50"], DEMO)
else:
    run_demo(["./rknn_paraformer_demo",
              "--encoder_path", "model/paraformer_encoder.rknn",
              "--decoder_path", "model/paraformer_decoder.rknn",
              "--tokens_path", "model/tokens.json",
              "--audio_path", wav], DEMO)

## Summary

- Non-autoregressive = all tokens in parallel, an order of magnitude faster
  than autoregressive ASR — great for realtime.
- Hotwords help with domain terms, names and other long-tail words.